# Signal Audit 2: Validation of the Signals Table

Three tests of `gld_player_signals_daily`:

| Part | Test | Sections |
|---|---|---|
| A | Does each column follow its formula in the workbook `gld-schemas.xlsx`? | 1 to 6 |
| B | The wrong signals, recomputed from every bet, against the table | 7 to 14 |
| C | How is the session length computed? | 15 |

Only aggregates are shown; personal columns are never read. **Run:** the first time with `RUN_S3 = True` (needs `aws sso login`); after that it reads the cache.

In [1]:
# reload the modules when they change, with no kernel restart
%reload_ext autoreload
%autoreload 2

import datetime as dt
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

RUN_S3 = True   # True the first time: reads S3 (needs `aws sso login`)
os.environ["RUN_S3"] = "1" if RUN_S3 else "0"

sys.path.insert(0, str(Path.cwd()))
sys.path.insert(0, str(Path.cwd().parent / "eda"))
import audit
import plots
import stats
import evidence as ev
import formula_check as fc
import recompute

pd.set_option("display.width", 220)
pd.set_option("display.max_colwidth", 120)
pd.set_option("display.max_rows", 300)
cfg = fc.load()                                                     # the formula checks (part A)
table_cfg = audit.load_config()                                     # the signals table
rc_cfg = audit.load_config(Path("metrics_recomputed.yaml"))         # our version (part B)
ks = fc.checks(cfg)
print(f"{ks['column'].nunique()} columns, {len(ks)} checks, snapshots {cfg['snapshots']}")
ks.groupby("kind").agg(checks=("column", "size"), columns=("column", "nunique"))

165 columns, 205 checks, snapshots ['2026-09-15', '2026-10-01']


,checks,columns
kind,,
band,3,3
bound,84,79
formula,18,18
identity,27,27
lag,14,14
none,25,25
validated,2,2
window,32,32


# Part A: The Workbook Formulas

Each of the 165 columns has its checks in `signal_checks.yaml`:

| Kind | What it checks |
|---|---|
| formula | the workbook formula on the other columns of the row |
| identity | an accounting rule (`ggr = wagered - won`) |
| window | a 7- or 30-day column is the sum of its daily values |
| lag | a `prior_*` column is the value of the window before |
| bound | an impossible value, or a longer window smaller than a shorter one |
| band | the bands follow the cutoffs [25, 50, 75] |
| none | key, text or metadata |

Some names differ from the workbook (for example, `recency_score` is `recency_pressure`, `monetary_score` is `value_decline`).

A value passes if it is within 0.01 + 0.1% of the expected one. A check **passes** if 99% of the rows pass, is **almost** right from 95%, and **fails** below. Two snapshot days.

## 1. Run the Checks

In [2]:
results = fc.run(cfg)
summary = results.pivot_table(index="kind", columns="verdict", values="column", aggfunc="count", fill_value=0)
summary

verdict,almost,fails,passes
kind,,,
bound,1,4,79
formula,0,2,16
identity,2,4,21
lag,0,4,10
window,0,0,32


## 2. Workbook Formulas and Accounting Identities

Worst first. *higher %* / *lower %*: rows where the table is above / below the expected value. A difference in one direction only means a different definition.

In [3]:
cols = ["column", "kind", "measure", "n", "pass %", "higher %", "lower %", "verdict", "note"]
show = lambda kinds: results[results["kind"].isin(kinds)].sort_values("pass %")[cols].round(2)
show(["formula", "identity"])

,column,kind,measure,n,pass %,higher %,lower %,verdict,note
14,bet_z_score,formula,M139,1788553,0.00,11.04,88.96,fails,"(avg_bet_eur[30d] - tenant_avg_bet) / tenant_stddev_bet, else 0. Tenant mean and deviation taken over the brand's pl..."
118,session_len_z,formula,M136,1788553,0.32,19.44,80.24,fails,"(avg_session_secs - tenant_avg_session) / tenant_stddev_session, else 0 (same approximation)"
71,ngr_eur_l180d,identity,M035,1788553,78.90,9.91,11.19,fails,NGR = GGR less bonus cost and game tax (M035); bonus cost taken as bonus GGR (hypothesis)
76,ngr_eur_l90d,identity,M035,1788553,86.94,2.74,10.32,fails,NGR = GGR less bonus cost and game tax (M035); bonus cost taken as bonus GGR (hypothesis)
112,rg_night_play_ratio,identity,NaN,1788553,91.46,0.56,7.98,fails,Same concept as tp_night_index (Signal Audit sheet)
73,ngr_eur_l30d,identity,M035,1788553,92.70,0.73,6.57,fails,NGR = GGR less bonus cost and game tax (M035); bonus cost taken as bonus GGR (hypothesis)
75,ngr_eur_l7d,identity,M035,1788553,96.35,0.24,3.41,almost,NGR = GGR less bonus cost and game tax (M035); bonus cost taken as bonus GGR (hypothesis)
77,ngr_eur_today,identity,M035,1788553,98.84,0.08,1.08,almost,NGR = GGR less bonus cost and game tax (M035); bonus cost taken as bonus GGR (hypothesis)
127,stake_escalation,formula,M140,1788553,100.00,0.00,0.00,passes,"(avg_bet_eur[7d] - avg_bet_eur[30d]) / avg_bet_eur[30d], else 0"
49,engagement_score,formula,M116,1788553,100.00,0.00,0.00,passes,"clamp(0,100, w1*(active_days[30d]/30)*100 + w2*min(1, sessions[30d]/60)*100 + w3*min(1, play_secs[30d]/108000)*100),..."


## 3. Windows and Lags

In [4]:
show(["window", "lag"])

,column,kind,measure,n,pass %,higher %,lower %,verdict,note
92,prior_betting_aggression_score,lag,NaN,1769798,49.36,20.91,29.73,fails,The score of 7 days before (lag assumed)
93,prior_churn_score,lag,NaN,1769798,84.04,6.96,9.00,fails,The score of 7 days before (lag assumed)
96,prior_engagement_score,lag,NaN,1769798,85.26,4.83,9.92,fails,The score of 7 days before (lag assumed)
99,prior_risk_score,lag,NaN,1769798,92.98,3.38,3.64,fails,The score of 7 days before (lag assumed)
72,ngr_eur_l30d,window,NaN,1788553,99.73,0.02,0.25,passes,Sum of ngr_eur_today over the 30 days up to the date
54,ggr_eur_l30d,window,NaN,1788553,99.93,0.03,0.05,passes,Sum of ggr_eur_today over the 30 days up to the date
103,real_ggr_eur_l30d,window,NaN,1788553,99.93,0.02,0.04,passes,Sum of real_ggr_eur_today over the 30 days up to the date
79,play_secs_l30d,window,NaN,1788553,99.95,0.05,0.00,passes,Sum of play_secs_today over the 30 days up to the date
150,wagered_eur_l30d,window,NaN,1788553,99.96,0.03,0.02,passes,Sum of wagered_eur_today over the 30 days up to the date
159,wagered_real_eur_l30d,window,NaN,1788553,99.96,0.02,0.01,passes,Sum of the daily value


## 4. Impossible Values and Broken Orders

Share of rows where the condition holds. Below 100% means some impossible values.

In [5]:
b = show(["bound"]).assign(**{"violations %": lambda t: 100 - t["pass %"]})
b[["column", "n", "pass %", "violations %", "verdict", "note"]]

,column,n,pass %,violations %,verdict,note
137,tp_hourly_entropy,1788553,88.97,11.03,fails,Unit 0-1 in the workbook
141,tp_weekend_index,1788553,90.64,9.36,fails,Unit 0-1 in the workbook
133,tp_avg_session_duration_secs,420533,92.70,7.30,fails,Recomputed in audit 2: sessions of 10 s to 4 h (session timeout not in the workbook). 0 means no time-of-play data a...
139,tp_median_session_duration_secs,420533,94.56,5.44,fails,Same range as the mean. 0 means no data and is left out
140,tp_night_index,1788553,96.82,3.18,almost,Unit 0-1 in the workbook
135,tp_avg_sessions_per_active_day,1788553,99.47,0.53,passes,At least one session on an active day
134,tp_avg_session_gap_hours,1788553,99.88,0.12,passes,Cannot be negative
126,sessions_with_bonus_l30d,1788553,100.00,0.00,passes,A part of the sessions
168,won_eur_l180d,1788553,100.00,0.00,passes,A longer window cannot be smaller (l90d to l180d)
173,won_eur_l90d,1788553,100.00,0.00,passes,A longer window cannot be smaller (l30d to l90d)


## 5. Bands

The score range of each band against the cutoffs [25, 50, 75].

In [6]:
bands = pd.concat([fc.band_checks(cfg, pd.Timestamp(d).date()) for d in cfg["snapshots"]])
bands.groupby(["column", "band", "bucket"], as_index=False).agg(rows=("rows", "sum"), score_min=("score_min", "min"),
                                                             score_max=("score_max", "max")).round(2)

,column,band,bucket,rows,score_min,score_max
0,churn_band,Churned,0,127,10.96,24.86
1,churn_band,Churned,1,8468,25.00,49.93
2,churn_band,Churned,2,475614,50.00,74.66
3,churn_band,Churned,3,8434,75.00,100.00
4,churn_band,High,2,45760,50.00,75.00
5,churn_band,Imminent,3,48203,75.00,100.00
6,churn_band,Low,0,22785,0.00,25.00
7,churn_band,Moderate,1,19201,25.00,50.00
8,churn_band,Not Scored,0,6243,0.00,24.72
9,churn_band,Not Scored,1,4482,25.00,49.97


## 6. Verdict per Column

The worst check of each column decides its verdict.

In [7]:
order = {"fails": 0, "almost": 1, "passes": 2, "no data": 3}
worst = (results.assign(rank=results["verdict"].map(order)).sort_values("rank")
         .groupby("column").agg(verdict=("verdict", "first"), checks=("kind", lambda k: ", ".join(sorted(set(k)))),
                                 worst_check=("note", "first"), worst_pass=("pass %", "first")))
try:
    import signal_validation as sv
    nb08 = pd.concat([r["table"].assign(brand=b) for b, r in sv.saved_results().items()])
    nb08 = nb08.groupby("column")["verdict"].agg(lambda v: "OK" if (v == "OK").all() else "fails").rename("notebook 08")
    worst = worst.join(nb08)
except FileNotFoundError:
    pass
none = ks.groupby("column")["kind"].agg(set)
not_checked = none[none.apply(lambda s: s <= {"none", "validated"})].index
band_only = none[none.apply(lambda s: s == {"band"})].index
print(f"{len(worst)} columns checked here, {len(band_only)} bands in section 5 ({', '.join(band_only)}), "
      f"{len(not_checked)} with no check of their own: {', '.join(not_checked)}")
display(worst["verdict"].value_counts())
worst.sort_values(["verdict", "worst_pass"]).round(2)

136 columns checked here, 3 bands in section 5 (churn_band, risk_band, value_tier), 26 with no check of their own: _ingested_at, _processed_at, _version, bonus_ggr_eur_l180d, bonus_ggr_eur_l90d, bonus_ggr_eur_today, brand_id, country, currency, has_sufficient_data, language, lifecycle_stage, lifetime_value_tier, param_version, player_id, prev_churn_band, prev_lifecycle_stage, prev_risk_band, prev_value_tier, snapshot_date, tax_eur_l180d, tax_eur_l90d, tax_eur_today, tenant_id, tenure_days, tp_window_days


verdict
passes    119
fails      14
almost      3
Name: count, dtype: int64

,verdict,checks,worst_check,worst_pass,notebook 08
column,,,,,
ngr_eur_l7d,almost,"identity, window",NGR = GGR less bonus cost and game tax (M035); bonus cost taken as bonus GGR (hypothesis),96.35,NaN
tp_night_index,almost,bound,Unit 0-1 in the workbook,96.82,fails
ngr_eur_today,almost,identity,NGR = GGR less bonus cost and game tax (M035); bonus cost taken as bonus GGR (hypothesis),98.84,NaN
bet_z_score,fails,formula,"(avg_bet_eur[30d] - tenant_avg_bet) / tenant_stddev_bet, else 0. Tenant mean and deviation taken over the brand's pl...",0.00,NaN
session_len_z,fails,formula,"(avg_session_secs - tenant_avg_session) / tenant_stddev_session, else 0 (same approximation)",0.32,NaN
prior_betting_aggression_score,fails,lag,The score of 7 days before (lag assumed),49.36,NaN
ngr_eur_l180d,fails,identity,NGR = GGR less bonus cost and game tax (M035); bonus cost taken as bonus GGR (hypothesis),78.90,NaN
prior_churn_score,fails,lag,The score of 7 days before (lag assumed),84.04,NaN
prior_engagement_score,fails,lag,The score of 7 days before (lag assumed),85.26,NaN


### Status of every column

Each of the 165 columns gets one status:

- **wrong:** it has an error of its own (`item` = its point in the error list);
- **affected:** its formula is right, but it uses a wrong column (see `expression`);
- **unconfirmed:** the workbook does not give the full definition;
- **correct:** it passes all its checks;
- **unchecked:** keys, text and metadata.

A column can pass its checks and still be wrong: `active_days_*` is the sum of its daily values, but the daily value uses the wrong definition.

In [8]:
status = ev.column_status()
display(status["status"].value_counts().rename("columns").to_frame().T)
for s in ("wrong", "affected", "unconfirmed"):
    print(s)
    display(status[status["status"] == s][["column", "checks", "worst %", "item", "expression"]].round(1))

status,correct,wrong,unchecked,affected,unconfirmed
columns,89,29,24,12,11


wrong


,column,checks,worst %,item,expression
10,brand_id,,NaN,it:brand,
48,active_days_l7d,"bound, window",100.0,it:activedays,
49,active_days_l30d,"bound, window",100.0,it:activedays,
50,active_days_l90d,bound,100.0,it:activedays,
51,active_days_l180d,bound,100.0,it:activedays,
52,active_days_today,bound,100.0,it:activedays,
63,play_secs_l7d,"bound, window",100.0,it:playsecs,
64,play_secs_l30d,"bound, window",99.9,it:playsecs,
65,play_secs_l90d,bound,100.0,it:playsecs,
66,play_secs_l180d,bound,100.0,it:playsecs,


affected


,column,checks,worst %,item,expression
119,engagement_score,formula,100.0,it:activedays,"least(100, greatest(0, 0.4 * (active_days_l30d / 30.0) * 100 + 0.3 * least(1, sessions_l30d / 60.0) * 100 + 0.3 * le..."
125,tp_business_hours_ratio,bound,100.0,it:tpdays,
131,tp_avg_sessions_per_active_day,bound,99.5,it:tpdays,
133,tp_max_sessions_single_day,bound,100.0,it:tpdays,
134,tp_window_days,,NaN,it:tpdays,
147,session_intensity_score,formula,100.0,it:playsecs,"CASE WHEN active_days_l30d > 0 THEN least(100, 100 * play_secs_today / (play_secs_l30d / active_days_l30d + 1)) END"
148,risk_score,formula,100.0,it:rgnight,"least(100, greatest(0, 20 * least(rg_loss_chasing_30d, 1) + 20 * least(rg_deposit_velocity, 1) + 15 * rg_night_play_..."
149,risk_band,band,NaN,it:rgnight,
151,frequency_decline,formula,100.0,it:activedays,"CASE WHEN active_days_l30d > 0 THEN 100 * greatest(0, 1 - (active_days_l7d / 7.0) / (active_days_l30d / 30.0)) END"
154,churn_score,formula,100.0,it:churnscore,"least(100, greatest(0, 0.50 * recency_pressure + 0.30 * frequency_decline + 0.20 * value_decline))"


unconfirmed


,column,checks,worst %,item,expression
18,ngr_eur_l7d,"identity, window",96.3,it:ngr,ggr_eur_l7d - bonus_ggr_eur_l7d - tax_eur_l7d
19,ngr_eur_l30d,"identity, window",92.7,it:ngr,ggr_eur_l30d - bonus_ggr_eur_l30d - tax_eur_l30d
71,ngr_eur_today,identity,98.8,it:ngr,ggr_eur_today - bonus_ggr_eur_today - tax_eur_today
75,ngr_eur_l90d,identity,86.9,it:ngr,ggr_eur_l90d - bonus_ggr_eur_l90d - tax_eur_l90d
79,ngr_eur_l180d,identity,78.9,it:ngr,ggr_eur_l180d - bonus_ggr_eur_l180d - tax_eur_l180d
120,prior_engagement_score,lag,85.3,it:prior,
137,bet_z_score,formula,0.0,it:z,None
138,session_len_z,formula,0.3,it:z,None
141,prior_betting_aggression_score,lag,49.4,it:prior,
150,prior_risk_score,lag,93.0,it:prior,


In [9]:
# correct and unchecked: every check passes (correct), or there is nothing to check (unchecked)
display(status[status["status"].isin(["correct", "unchecked"])].sort_values(["status", "column"])[["column", "status", "checks", "worst %"]].round(1))

,column,status,checks,worst %
104,avg_bet_eur_l30d,correct,identity,100.0
103,avg_bet_eur_l7d,correct,identity,100.0
61,bets_l180d,correct,bound,100.0
59,bets_l30d,correct,"bound, window",100.0
58,bets_l7d,correct,"bound, window",100.0
60,bets_l90d,correct,bound,100.0
62,bets_today,correct,bound,100.0
140,betting_aggression_score,correct,formula,100.0
38,bonus_ggr_eur_l30d,correct,window,100.0
37,bonus_ggr_eur_l7d,correct,window,100.0


# Part B: The Signals Recomputed From Every Bet

I recompute the wrong signals myself and compare them with the table.

- **Sessions:** the bets of a player with no gap longer than **30 minutes** (15 and 60 are also tested). Duration = last bet − first bet. Sessions of **one bet** are counted apart.
- **Night:** 00:00 to 06:00 in the local time of the currency.
- **Active days and hold:** from `gld_player_gaming_daily`.

Definitions: `metrics_recomputed.yaml`. Days are built one at a time and cached.

## 7. Is `accountId` the `player_id` of Gold?

Before joining anything: are the player ids of the transactions and of gold the same?

In [10]:
keys = recompute.key_check(rc_cfg, dt.date(2026, 9, 15))
keys.assign(**{"share of gold in landing %": 100 * keys["in_both"] / keys["gold_players"],
               "share of landing in gold %": 100 * keys["in_both"] / keys["landing_players"]}).round(2)

,brand_id,landing_players,gold_players,in_both,share of gold in landing %,share of landing in gold %
0,14,1815,1823,0,0.00,0.00
1,23,6310,6204,2,0.03,0.03
2,64,2500,2468,0,0.00,0.00
3,72,2526,2460,0,0.00,0.00
4,73,5123,5077,0,0.00,0.00


## 8. Build the Sessions

One day at a time. Days already built are skipped.

In [11]:
built = recompute.build(rc_cfg)
print(f"built {len(built)} new day(s); {len(audit.period_days(rc_cfg))} days in the period")

built 0 new day(s); 30 days in the period


### Time of play

Stake by local hour and day of the week, for the weekend share and the hourly entropy.

In [12]:
built = recompute.build_time_of_play(rc_cfg)
print(f"built {len(built)} new day(s) of time of play")

built 0 new day(s) of time of play


## 9. Scorecard of the Recomputed Signals

The same checks as in notebook 01.

In [14]:
rc_prof, rc_res, card = {}, {}, []
for name in rc_cfg["metrics"]:
    prof, hist = audit.metric_data(rc_cfg, name)
    res = stats.fit(rc_cfg, name, prof, hist)
    rc_prof[name], rc_res[name] = prof, res
    card.append(stats.summary_row(rc_cfg, name, prof, res))
rc_card = pd.DataFrame(card).set_index("metric")
rc_card.style.format({"rows": "{:,}", "null %": "{:.1f}", "p1": "{:.3g}", "median": "{:.3g}", "p99": "{:.3g}",
                      "max": "{:.3g}", "outside range %": "{:.2f}", "KS distance": "{:.3f}"})

,table,expected,rows,null %,p1,median,p99,max,outside range %,peaks,best fit,KS distance,fit
metric,,,,,,,,,,,,,
rc_session_duration_secs,rc_sessions,lognormal,"980,101",0.0,12.3,1.52e+03,1.88e+04,1.09e+05,2.27,1,Log-normal,0.063,acceptable
rc_session_duration_by_timeout,rc_sessions,lognormal,"2,958,935",0.0,11.9,1.45e+03,2.02e+04,1.09e+05,2.62,1,Log-normal,0.055,acceptable
rc_player_avg_session_secs,rc_player_month,lognormal,"97,964",0.0,30.8,1.51e+03,1.03e+04,4.18e+04,0.30,1,Log-normal,0.077,acceptable
rc_bets_per_session,rc_sessions,count,"1,127,438",0.0,1,110,3.79e+03,4.61e+04,0.00,2,Log-normal,0.135,different
rc_sessions_per_active_day,rc_player_day,count,"537,964",0.0,1,1,9,22,0.00,1,Negative binomial,0.006,close
rc_single_bet_share,rc_player_day,share,"537,964",0.0,0,0,1,1,0.00,2,,nan,
rc_session_gap_hours,rc_session_gaps,lognormal,"1,026,275",0.0,0.517,5.12,266,709,0.00,2,Log-normal,0.129,different
rc_night_share,rc_player_month,share,"100,038",0.0,0,0.0477,1,1,0.00,2,,nan,
rc_hold_pct_month,gaming_player_month,share,"100,041",0.0,-63.6,18.8,100,100,0.25,2,,nan,


## 10. The Table against Our Version

Both versions on the same plot. Shaded area: outside the real-life range.

In [15]:
tb_prof, tb_res = {}, {}
for name in {m["compare_to"] for m in rc_cfg["metrics"].values() if m.get("compare_to")}:
    prof, hist = audit.metric_data(table_cfg, name)
    tb_prof[name], tb_res[name] = prof, stats.fit(table_cfg, name, prof, hist)

rows = []
for name, m in rc_cfg["metrics"].items():
    if not m.get("compare_to"):
        continue
    t = m["compare_to"]
    plots.compare_results([("signals table: " + t, table_cfg, tb_res[t]), ("recomputed: " + name, rc_cfg, rc_res[name])],
                       title=f"{t} against {name}")
    plt.show()
    for label, cfg, n, p, r in (("table", table_cfg, t, tb_prof[t], tb_res[t]), ("recomputed", rc_cfg, name, rc_prof[name], rc_res[name])):
        row = stats.summary_row(cfg, n, p, r)
        rows.append({"signal": t, "version": label, "metric": n, "rows": row["rows"], "median": row["median"],
                     "p99": row["p99"], "max": row["max"], "outside range %": row["outside range %"],
                     "peaks": row["peaks"], "best fit": row["best fit"], "KS": row["KS distance"]})
compare = pd.DataFrame(rows).set_index(["signal", "version", "metric"])
compare.style.format({"rows": "{:,}", "median": "{:.3g}", "p99": "{:.3g}", "max": "{:.3g}",
                      "outside range %": "{:.2f}", "KS": "{:.3f}"})

## 11. Each Recomputed Signal

In [16]:
for name in rc_cfg["metrics"]:
    plots.metric(rc_cfg, rc_res[name], rc_prof[name])
    plt.show()

/home/javier/whizdom-churn/signal_audit/audit.py:478: RuntimeWarning: More than 20 figures have been opened. Figures created through the pyplot interface (`matplotlib.pyplot.figure`) are retained until explicitly closed and may consume too much memory. (To control this warning, see the rcParam `figure.max_open_warning`). Consider using `matplotlib.pyplot.close()`.
  ax = ax or plt.subplots(figsize=(8.5, 3.8))[1]


## 12. How Much the Session Timeout Matters

Sessions with a timeout of 15, 30 and 60 minutes. If the shape barely changes, the choice of 30 minutes does not matter much.

In [17]:
name = "rc_session_duration_by_timeout"
seg = audit.segment_histograms(rc_cfg, name, "timeout_min")
m = rc_cfg["metrics"][name]
rows = []
for t, g in seg.groupby("segment"):
    h = stats.complete(g[["bin", "count"]], m)
    rows.append({"timeout (min)": int(t), "sessions of 2 bets or more": int(h["count"].sum()),
                 "median (s)": stats.hist_quantile(h, 0.5), "p99 (s)": stats.hist_quantile(h, 0.99),
                 "above 4 h %": 100 * h.loc[h["lo"] >= 14400, "count"].sum() / h["count"].sum()})
display(pd.DataFrame(rows).sort_values("timeout (min)").round(1))
plots.segments(rc_cfg, name, seg)
plt.show()

,timeout (min),sessions of 2 bets or more,median (s),p99 (s),above 4 h %
0,15,1141669,1258.9,15848.9,0.8
1,30,980101,1584.9,19952.6,1.7
2,60,837165,2238.7,28183.8,3.9


## 13. Sessions of One Bet

The table gives these sessions a length of exactly 1 second. Here they are counted apart.

In [18]:
for name in ("rc_bets_per_session", "rc_single_bet_share"):
    for segment in rc_cfg["metrics"][name].get("segments", []):
        print(f"{name} by {segment}")
        display(stats.segment_peaks(rc_cfg, name, audit.segment_histograms(rc_cfg, name, segment)))
print(stats.point_masses(rc_cfg, "rc_bets_per_session", audit.metric_data(rc_cfg, "rc_bets_per_session")[1]).round(2))

rc_bets_per_session by brand_id


,segment,rows,peaks,peak positions
0,23,399172,2,"2.99, 335"
1,73,347242,2,"2.11, 211"
2,72,135925,1,237
3,64,135388,1,266
4,14,109711,1,266


rc_single_bet_share by brand_id


,segment,rows,peaks,peak positions
0,23,189559,1,0.01
1,73,155670,2,"0.01, 1.01"
2,64,71039,1,0.01
3,72,70648,1,0.01
4,14,51048,1,0.01


   value  share %  exact value
0      1    13.07         True
1      2     4.81         True
2      3     2.49         True
3      4     1.61         True
4      5     1.14         True
5     10     1.05        False


## 14. Night Share by Brand and Currency

The recomputed night share must stay between 0 and 1.

In [19]:
for segment in rc_cfg["metrics"]["rc_night_share"]["segments"]:
    seg = audit.segment_histograms(rc_cfg, "rc_night_share", segment)
    display(stats.outside_by_segment(rc_cfg, "rc_night_share", seg).round(2))
    plots.segments(rc_cfg, "rc_night_share", seg)
    plt.show()

,segment,rows,below range %,above range %
0,73,28290,0.0,0.0
1,23,27709,0.0,0.0
2,72,20156,0.0,0.0
3,64,15642,0.0,0.0
4,14,8241,0.0,0.0


,segment,rows,below range %,above range %
0,TRY,82432,0.0,0.0
1,EUR,13723,0.0,0.0
2,PLN,3750,0.0,0.0
3,AUD,71,0.0,0.0
4,INR,34,0.0,0.0
5,USDT,25,0.0,0.0
6,CAD,1,0.0,0.0
7,NZD,1,0.0,0.0
8,USD,1,0.0,0.0


# Part C: Session Length

## 15. How Is the Session Length Computed?

The workbook does not give the formula. I test two things with the other columns of the row:

1. **Where it comes from:** is it play time / number of sessions?
2. **Rules a real session must follow:** no session over one day, no day with more than 24 hours of sessions or play, no negative gap between sessions.

In [20]:
checks = ev.session_checks(table_cfg)
display(checks.pivot_table(index=["check", "what it tests"], columns="brand", values="holds %", sort=False, dropna=False).round(2))

brand                                                                                                                           14      23     64     72     73    all
check                                    what it tests                                                                                                                
mean = play_secs / sessions (30 days)    The mean session is the play time of 30 days over the sessions of 30 days            2.05    2.66   2.15   2.12   4.27   2.90
                                         The same with 7 days                                                                  NaN     NaN    NaN    NaN    NaN    NaN
                                         Only one session in the window, or every session the same length                      NaN     NaN    NaN    NaN    NaN    NaN
                                         Rare for a right-skewed length (log-normal): the median is usually below the mean     NaN     NaN    NaN    NaN    NaN    NaN
                                         Sessions of one event: the start equals the end                                       NaN     NaN    NaN    NaN    NaN    NaN
                                         Of the 1-second rows: a single session                                                NaN     NaN    NaN    NaN    NaN    NaN
                                         A session that never closes                                                           NaN     NaN    NaN    NaN    NaN    NaN
                                         The sessions of one day last longer than the day                                      NaN     NaN    NaN    NaN    NaN    NaN
                                         One session longer than the whole window it is computed on                            NaN     NaN    NaN    NaN    NaN    NaN
                                         Negative gap: a session starts before the previous one ends                           NaN     NaN    NaN    NaN    NaN    NaN
                                         More play time than the day has                                                       NaN     NaN    NaN    NaN    NaN    NaN
                                         More play time than the 30 days have                                                  NaN     NaN    NaN    NaN    NaN    NaN
mean = play_secs / sessions (7 days)     The mean session is the play time of 30 days over the sessions of 30 days             NaN     NaN    NaN    NaN    NaN    NaN
                                         The same with 7 days                                                                 1.32    1.30   1.56   1.34   2.83   1.86
                                         Only one session in the window, or every session the same length                      NaN     NaN    NaN    NaN    NaN    NaN
                                         Rare for a right-skewed length (log-normal): the median is usually below the mean     NaN     NaN    NaN    NaN    NaN    NaN
                                         Sessions of one event: the start equals the end                                       NaN     NaN    NaN    NaN    NaN    NaN
                                         Of the 1-second rows: a single session                                                NaN     NaN    NaN    NaN    NaN    NaN
                                         A session that never closes                                                           NaN     NaN    NaN    NaN    NaN    NaN
                                         The sessions of one day last longer than the day                                      NaN     NaN    NaN    NaN    NaN    NaN
                                         One session longer than the whole window it is computed on                            NaN     NaN    NaN    NaN    NaN    NaN
                                         Negative gap: a session starts before the previous one ends                           NaN     NaN    NaN    NaN    NaN

**Mean session / (play_secs / sessions):** 1 would mean the session comes from play time. Also the window of the time-of-play columns (`tp_window_days`).

In [21]:
display(ev.session_ratio(table_cfg).round(3))
w = ev.window_days(table_cfg)
fig, ax = plt.subplots(figsize=(10, 3))
ax.bar(w["window_days"], 100 * w["rows"] / w["rows"].sum(), color="#8fb3c7")
ax.set(title="tp_window_days of the rows with a session length", xlabel="days", ylabel="% of rows")
plt.show()

,brand,rows,ratio p1,ratio p10,ratio p25,ratio p50,ratio p75,ratio p90,ratio p99,window_days p1,window_days p10,window_days p25,window_days p50,window_days p75,window_days p90,window_days p99
0,14,236805.0,0.307,0.994,1.457,2.448,4.396,8.986,130.000,90.0,90.0,90.0,90.0,90.0,90.0,90.0
1,23,386753.0,0.290,0.901,1.418,2.447,4.472,9.743,131.081,90.0,90.0,90.0,90.0,90.0,90.0,90.0
2,64,299076.0,0.422,1.193,1.733,2.780,4.995,11.102,158.333,90.0,90.0,90.0,90.0,90.0,90.0,90.0
3,72,542531.0,0.416,1.310,2.056,3.523,7.266,19.530,311.000,90.0,90.0,90.0,90.0,90.0,90.0,90.0
4,73,694032.0,0.197,0.724,1.160,2.498,5.293,13.261,257.125,90.0,90.0,90.0,90.0,90.0,90.0,90.0
5,all,2159197.0,0.283,0.942,1.538,2.757,5.419,13.198,219.750,90.0,90.0,90.0,90.0,90.0,90.0,90.0


## 16. Findings

### A. The workbook formulas

- **The table follows its formulas almost everywhere:** 158 of 175 checks pass. Scores, windows, accounting rules and bands are right.
- **The real errors are the time-of-play columns (`tp_*`):** session, night, weekend and entropy.
- **A right formula on a wrong input is still wrong:** `churn_score` and `deposit_recency_score` use `days_since_bet`.
- **Cannot be checked:** NGR, `prior_*_score` and the z-scores; the workbook does not give the full definition.

### B. The signals recomputed from every bet

- **Session length:** 0.3% outside 10 s to 4 h per player (8.4% in the table); the median is about the same (25 against 28 minutes). Some recomputed sessions are still long (up to 30 h of non-stop play, probably automatic).
- **One-bet sessions** are 13% of all sessions; the table gives them 1 second. The timeout (15, 30 or 60 minutes) changes the median, not the shape.
- **Night, weekend and entropy** recomputed are always between 0 and 1.
- **Sessions per day and hold match the table.**
- **`accountId` is not the gold `player_id`** (2 of about 18,000 match), so the comparison is by distribution, not player by player, until the key is confirmed.

### C. How the session length is computed

- **The window is 90 days**, not 30.
- **It is not play time / sessions** (only 3% match): the table measures sessions from start to end.
- **Exactly 1 s = a session of one event.**
- **Some sessions never close** (0.6% over 1 day) and **some overlap** (0.9% negative gaps).
- **`play_secs` is also impossible:** 3.7% of the days have more than 24 hours, probably because parallel games are added up.